In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset


# ============================================================
# 1. Configuration
# ============================================================

V = 256                 # byte-level vocabulary
context_length = 128

batch_size = 16

d_model = 128
n_heads = 4
d_head = d_model // n_heads
d_ff = 512
n_layers = 4

assert d_model % n_heads == 0

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("device:", device)


# ============================================================
# 2. RoPE
# ============================================================

def apply_rope(x, base=10000.0):
    """
    x: (B, H, n, d_head)

    RoPE applies a position-dependent 2D rotation
    to adjacent feature pairs.

    Returns:
        (B, H, n, d_head)
    """

    B, H, n, d_head = x.shape

    assert d_head % 2 == 0

    # One frequency per adjacent pair
    freq = 1.0 / (
        base ** (
            torch.arange(
                0,
                d_head,
                2,
                device=x.device,
                dtype=x.dtype
            ) / d_head
        )
    )  # (d_head / 2,)

    positions = torch.arange(
        n,
        device=x.device,
        dtype=x.dtype
    )  # (n,)

    # (n, d_head/2)
    angles = positions[:, None] * freq[None, :]

    # Broadcast over batch and heads
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]

    # (B,H,n,d_head) -> (B,H,n,d_head/2,2)
    x_pairs = x.reshape(
        B,
        H,
        n,
        d_head // 2,
        2
    )

    x_even = x_pairs[..., 0]   # (B,H,n,d_head/2)
    x_odd  = x_pairs[..., 1]

    # Equivalent to applying a 2D rotation matrix
    # to every adjacent feature pair.
    x_rot = torch.stack(
        [
            x_even * cos - x_odd * sin,
            x_even * sin + x_odd * cos,
        ],
        dim=-1
    )

    # (B,H,n,d_head/2,2) -> (B,H,n,d_head)
    return x_rot.flatten(-2)


# ============================================================
# 3. Multi-head causal self-attention
# ============================================================

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)

        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        """
        x: (B, n, d_model)
        """

        B, n, _ = x.shape

        # ----------------------------------
        # Q / K / V projection
        # ----------------------------------

        q = self.Wq(x)  # (B,n,d_model)
        k = self.Wk(x)
        v = self.Wv(x)

        # ----------------------------------
        # Split model width into heads
        # ----------------------------------

        q = q.view(B, n, self.n_heads, self.d_head)
        k = k.view(B, n, self.n_heads, self.d_head)
        v = v.view(B, n, self.n_heads, self.d_head)

        # (B,n,H,d_head) -> (B,H,n,d_head)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ----------------------------------
        # RoPE on Q and K
        # ----------------------------------

        q = apply_rope(q)
        k = apply_rope(k)

        # ----------------------------------
        # Scaled dot-product attention
        # ----------------------------------

        # (B,H,n,d_head) @ (B,H,d_head,n)
        # -> (B,H,n,n)
        scores = q @ k.transpose(-2, -1)

        scores = scores / (self.d_head ** 0.5)

        # ----------------------------------
        # Causal mask
        # ----------------------------------

        mask = torch.triu(
            torch.ones(
                n,
                n,
                dtype=torch.bool,
                device=x.device
            ),
            diagonal=1
        )

        scores = scores.masked_fill(
            mask,
            float("-inf")
        )

        # Distribution over key positions
        attn = torch.softmax(
            scores,
            dim=-1
        )  # (B,H,n,n)

        # ----------------------------------
        # Retrieve values
        # ----------------------------------

        out = attn @ v
        # (B,H,n,d_head)

        # ----------------------------------
        # Merge heads
        # ----------------------------------

        out = out.transpose(1, 2)
        # (B,n,H,d_head)

        out = out.reshape(
            B,
            n,
            self.d_model
        )
        # (B,n,d_model)

        out = self.Wo(out)

        return out


# ============================================================
# 4. MLP
# ============================================================

class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            d_ff
        )

        self.fc2 = nn.Linear(
            d_ff,
            d_model
        )

    def forward(self, x):
        # (B,n,d_model)
        x = self.fc1(x)

        # (B,n,d_ff)
        x = F.silu(x)

        # (B,n,d_model)
        x = self.fc2(x)

        return x


# ============================================================
# 5. Pre-norm decoder block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        d_ff
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = SelfAttention(
            d_model,
            n_heads
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = MLP(
            d_model,
            d_ff
        )

    def forward(self, x):

        # Attention learned update + residual
        x = x + self.attn(
            self.norm1(x)
        )

        # MLP learned update + residual
        x = x + self.mlp(
            self.norm2(x)
        )

        return x


# ============================================================
# 6. Tiny decoder-only language model
# ============================================================

class TinyDecoderLM(nn.Module):
    def __init__(
        self,
        V,
        d_model,
        n_heads,
        d_ff,
        n_layers
    ):
        super().__init__()

        # (V, d_model)
        self.embedding = nn.Embedding(
            V,
            d_model
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model,
                n_heads,
                d_ff
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(
            d_model
        )

        # d_model -> vocabulary logits
        self.lm_head = nn.Linear(
            d_model,
            V,
            bias=False
        )

    def forward(self, token_ids):
        """
        token_ids: (B,n)

        returns:
            logits: (B,n,V)
        """

        # (B,n) -> (B,n,d_model)
        x = self.embedding(token_ids)

        for block in self.blocks:
            x = block(x)

        # (B,n,d_model)
        x = self.final_norm(x)

        # (B,n,V)
        logits = self.lm_head(x)

        return logits


# ============================================================
# 7. Load TinyStories subset
# ============================================================

ds = load_dataset(
    "karpathy/tinystories-gpt4-clean",
    split="train"
)

# Keep train / validation disjoint
val_stories = ds.select(
    range(10_000, 11_000)
)

train_stories = ds.select(
    range(20_000, 40_000)
)

train_texts = train_stories["text"]
val_texts = val_stories["text"]

print("train stories:", len(train_texts))
print("val stories:", len(val_texts))


# ============================================================
# 8. Fixed-length causal LM batch sampler
# ============================================================

def sample_batch(
    stories,
    batch_size,
    context_length,
    device=device
):
    """
    Pick batch_size random stories.

    For each story:
        sample context_length + 1 consecutive bytes

        x = first context_length
        y = same sequence shifted by one

    Returns:
        x: (B, context_length)
        y: (B, context_length)
    """

    # Fine for our small dataset.
    # We can pre-tokenize later for speed.
    valid = [
        story
        for story in stories
        if len(story.encode("utf-8"))
        >= context_length + 1
    ]

    xs = []
    ys = []

    for _ in range(batch_size):

        story_idx = torch.randint(
            0,
            len(valid),
            (1,)
        ).item()

        story = valid[story_idx]

        tokens = torch.tensor(
            list(story.encode("utf-8")),
            dtype=torch.long
        )

        # Need room for context_length + 1 bytes.
        max_start = (
            len(tokens)
            - context_length
        )

        start = torch.randint(
            0,
            max_start,
            (1,)
        ).item()

        chunk = tokens[
            start:
            start + context_length + 1
        ]

        # shifted causal LM pair
        x = chunk[:-1]
        y = chunk[1:]

        xs.append(x)
        ys.append(y)

    x = torch.stack(xs).to(device)
    y = torch.stack(ys).to(device)

    return x, y


# ============================================================
# 9. Test data pipeline
# ============================================================

x_batch, y_batch = sample_batch(
    train_texts,
    batch_size=batch_size,
    context_length=context_length
)

print()
print("x:", x_batch.shape)
print("y:", y_batch.shape)
print("dtype:", x_batch.dtype)
print(
    "token range:",
    x_batch.min().item(),
    x_batch.max().item()
)

assert x_batch.shape == (
    batch_size,
    context_length
)

assert y_batch.shape == (
    batch_size,
    context_length
)

assert x_batch.dtype == torch.long


# Visually verify shift
print()
print("INPUT:")
print(
    bytes(
        x_batch[0].cpu().tolist()
    ).decode(
        "utf-8",
        errors="replace"
    )
)

print("\nTARGET:")
print(
    bytes(
        y_batch[0].cpu().tolist()
    ).decode(
        "utf-8",
        errors="replace"
    )
)


# ============================================================
# 10. Instantiate model
# ============================================================

torch.manual_seed(0)

model = TinyDecoderLM(
    V=V,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    n_layers=n_layers
).to(device)


num_params = sum(
    p.numel()
    for p in model.parameters()
)

print()
print(
    f"parameters: {num_params:,}"
)


# ============================================================
# 11. Test one forward pass
# ============================================================

with torch.no_grad():
    logits = model(x_batch)

print(
    "logits:",
    logits.shape
)

assert logits.shape == (
    batch_size,
    context_length,
    V
)


# ============================================================
# 12. Initial random-model loss
# ============================================================

pred = logits.reshape(
    -1,
    V
)

gold = y_batch.reshape(-1)

loss = F.cross_entropy(
    pred,
    gold
)

print(
    "initial loss:",
    loss.item()
)

print(
    "uniform baseline log(V):",
    torch.log(
        torch.tensor(float(V))
    ).item()
)

/Users/baraa/Library/Caches/pypoetry/virtualenvs/transformers-from-first-principles-ROJJTl4v-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


Generating train split: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2732634/2732634 [00:01<00:00, 1660359.20 examples/s]


train stories: 20000
val stories: 1000

x: torch.Size([16, 128])
y: torch.Size([16, 128])
dtype: torch.int64
token range: 10 121

INPUT:
I found a nail. Can I help you fix the fence?" His dad smiled and said, "Yes, Tim. You are ready to help me. Let's gain some mor

TARGET:
 found a nail. Can I help you fix the fence?" His dad smiled and said, "Yes, Tim. You are ready to help me. Let's gain some more

parameters: 856,832
logits: torch.Size([16, 128, 256])
initial loss: 5.670619010925293
uniform baseline log(V): 5.545177459716797


In [2]:
gold.shape

torch.Size([2048])

In [3]:
2048 // 16

128